<h1><center>Laboratorio 6: Optimización de modelos 🧪</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Primavera 2025</strong></center>

### Cuerpo Docente:

- Profesores: Diego Cortez, Gabriel Iturra
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Nicolás Cabello, Cristopher Urbina

### Equipo:

- Nombre de alumno 1: Naomí Cautivo B.
- Nombre de alumno 2: Máximo Flores Valenzuela


Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda **fuertemente** asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

### **Link de repositorio de GitHub:** [maxfloresv/MDS7202](https://github.com/maxfloresv/MDS7202)

### Temas a tratar

- Predicción de demanda usando `xgboost`
- Búsqueda del modelo óptimo de clasificación usando `optuna`
- Uso de pipelines.


### Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: 6 días de plazo con descuento de 1 punto por día. Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda fuertemente asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

# El emprendimiento de Fiu

Tras liderar de manera exitosa la implementación de un proyecto de ciencia de datos para caracterizar los datos generados en Santiago 2023, el misterioso corpóreo **Fiu** se anima y decide levantar su propio negocio de consultoría en machine learning. Tras varias e intensas negociaciones, Fiu logra encontrar su *primera chamba*: predecir la demanda (cantidad de venta) de una famosa productora de bebidas de calibre mundial. Al ver el gran potencial y talento que usted ha demostrado en el campo de la ciencia de datos, Fiu lo contrata como data scientist para que forme parte de su nuevo emprendimiento.

Para este laboratorio deben trabajar con los datos `sales.csv` subidos a u-cursos, el cual contiene una muestra de ventas de la empresa para diferentes productos en un determinado tiempo.

Para comenzar, cargue el dataset señalado y visualice a través de un `.head` los atributos que posee el dataset.

<i><p align="center">Fiu siendo felicitado por su excelente desempeño en el proyecto de caracterización de datos</p></i>
<p align="center">
  <img src="https://media-front.elmostrador.cl/2023/09/A_UNO_1506411_2440e.jpg">
</p>

In [50]:
import pandas as pd
import numpy as np
from datetime import datetime

df = pd.read_csv('sales.csv')
df.head()

,id,date,city,lat,long,pop,shop,brand,container,capacity,price,quantity
0,0,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,glass,500ml,0.96,13280
1,1,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,plastic,1.5lt,2.86,6727
2,2,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,can,330ml,0.87,9848
3,3,31/01/12,Athens,37.97945,23.71622,672130,shop_1,adult-cola,glass,500ml,1.00,20050
4,4,31/01/12,Athens,37.97945,23.71622,672130,shop_1,adult-cola,can,330ml,0.39,25696


## 1 Generando un Baseline (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/O-lan6TkadUAAAAC/what-i-wnna-do-after-a-baseline.gif">
</p>

Antes de entrenar un algoritmo, usted recuerda los apuntes de su magíster en ciencia de datos y recuerda que debe seguir una serie de *buenas prácticas* para entrenar correcta y debidamente su modelo. Después de un par de vueltas, llega a las siguientes tareas:

1. Separe los datos en conjuntos de train (70%), validation (20%) y test (10%). Fije una semilla para controlar la aleatoriedad. [0.5 puntos]
2. Implemente un `FunctionTransformer` para extraer el día, mes y año de la variable `date`. Guarde estas variables en el formato categorical de pandas. [1 punto]
3. Implemente un `ColumnTransformer` para procesar de manera adecuada los datos numéricos y categóricos. Use `OneHotEncoder` para las variables categóricas. `Nota:` Utilice el método `.set_output(transform='pandas')` para obtener un DataFrame como salida del `ColumnTransformer` [1 punto]
4. Guarde los pasos anteriores en un `Pipeline`, dejando como último paso el regresor `DummyRegressor` para generar predicciones en base a promedios. [0.5 punto]
5. Entrene el pipeline anterior y reporte la métrica `mean_absolute_error` sobre los datos de validación. ¿Cómo se interpreta esta métrica para el contexto del negocio? [0.5 puntos]
6. Finalmente, vuelva a entrenar el `Pipeline` pero esta vez usando `XGBRegressor` como modelo **utilizando los parámetros por default**. ¿Cómo cambia el MAE al implementar este algoritmo? ¿Es mejor o peor que el `DummyRegressor`? [1 punto]
7. Guarde ambos modelos en un archivo .pkl (uno cada uno) [0.5 puntos]

In [51]:
from sklearn import set_config
from sklearn.model_selection import train_test_split
set_config(transform_output="pandas")

RANDOM_STATE = 42

# 1.
X = df.drop(columns=["quantity"])
y = df["quantity"]

X_train_val, X_test, y_train_val, y_test = train_test_split(
  X, y, test_size=0.1, random_state=RANDOM_STATE
)

# 90 % * x = 20 % => x = 2 / 9
X_train, X_val, y_train, y_val = train_test_split(
  X_train_val, y_train_val, test_size=2 / 9, random_state=RANDOM_STATE
)

print(f"Train %: {X_train.shape[0] / X.shape[0] * 100:.2f}%")
print(f"Validation %: {X_val.shape[0] / X.shape[0] * 100:.2f}%")
print(f"Test %: {X_test.shape[0] / X.shape[0] * 100:.2f}%")

Train %: 69.98%
Validation %: 20.01%
Test %: 10.01%


In [52]:
import joblib
import os
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyRegressor

os.makedirs("models", exist_ok=True)

def split_date(X):
  """
  Split the "date" column into three new columns: "day", "month", and "year".

  Parameters
  ----------
  X : pd.DataFrame
    Input dataframe with a "date" column.

  Returns
  -------
  pd.DataFrame
    Dataframe with "day", "month", and "year" columns instead of "date".
  """
  X = X.copy()
  X["date"] = pd.to_datetime(X["date"], format="%d/%m/%y")
  X["day"] = (X["date"].dt.day).astype('category')
  X["month"] = (X["date"].dt.month).astype('category')
  X["year"] = (X["date"].dt.year).astype('category')
  X = X.drop(columns=["date"])
  return X

categorical = ['city', 'day', 'month', 'year', 'shop', 'brand', 'container', 'capacity']

if not os.path.exists("models/dummy_pipeline.pkl"):
  # 3.

  # drop='first' to avoid perfect multicollinearity.
  # sparse_output=False to get a DataFrame output instead of a sparse matrix.
  ct = ColumnTransformer(transformers=[
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False), categorical),
  ], remainder="passthrough")

  # 4.
  pipeline = Pipeline(steps=[
    # 2.
    ("function_transformer", FunctionTransformer(func=split_date)),
    ("column_transformer", ct),
    # By default, DummyRegressor uses the mean to predict.
    ("regressor", DummyRegressor()),
  ])

  pipeline.fit(X_train, y_train)
  # 7.
  joblib.dump(pipeline, "models/dummy_pipeline.pkl")
else:
  pipeline = joblib.load("models/dummy_pipeline.pkl")
  print("Loaded dummy pipeline from file.")

Loaded dummy pipeline from file.


In [53]:
from sklearn.metrics import mean_absolute_error

# 5.
y_pred_val = pipeline.predict(X_val)
mae = mean_absolute_error(y_val, y_pred_val)
print(f"Validation MAE: {mae:.2f}")

Validation MAE: 13543.96


5. Entrene el pipeline anterior y reporte la métrica `mean_absolute_error` sobre los datos de validación. ¿Cómo se interpreta esta métrica para el contexto del negocio?
> **Respuesta**: Para el negocio, esta métrica significa que el error promedio de predicción del modelo es, en cantidad (`quantity`), en torno a $13.543$ unidades tanto para abajo como para arriba, recordando que la definición de MAE es la siguiente:
> $$
> \text{MAE}(\vec{y}, \vec{\hat{y}}) = \frac{1}{N} \sum_{i=1}^N \lvert \hat{y}_i - y_i \rvert
> $$

In [54]:
from xgboost import XGBRegressor

if not os.path.exists("models/xgb_pipeline.pkl"):
  # 6.
  pipeline = Pipeline(steps=[
    ("function_transformer", FunctionTransformer(func=split_date)),
    ("column_transformer", ct),
    ("regressor", XGBRegressor()),
  ])

  pipeline.fit(X_train, y_train)
  # 7.
  joblib.dump(pipeline, "models/xgb_pipeline.pkl")
else:
  pipeline = joblib.load("models/xgb_pipeline.pkl")
  print("Loaded xgb pipeline from file.")

Loaded xgb pipeline from file.


In [55]:
y_pred_val = pipeline.predict(X_val)
mae = mean_absolute_error(y_val, y_pred_val)
print(f"Validation MAE: {mae:.2f}")

Validation MAE: 2483.77


6. Finalmente, vuelva a entrenar el `Pipeline` pero esta vez usando `XGBRegressor` como modelo **utilizando los parámetros por default**. ¿Cómo cambia el MAE al implementar este algoritmo? ¿Es mejor o peor que el `DummyRegressor`?

> **Respuesta**: El MAE baja de $13.543$ (`DummyRegressor`) a $2.483$ (`XGBRegressor`), que es una baja considerable en el contexto del problema. Con esto en mente, las predicciones de `XGBRegressor` son mejores que las de `DummyRegressor`.

## 2. Forzando relaciones entre parámetros con XGBoost (10 puntos)

<p align="center">
  <img src="https://64.media.tumblr.com/14cc45f9610a6ee341a45fd0d68f4dde/20d11b36022bca7b-bf/s640x960/67ab1db12ff73a530f649ac455c000945d99c0d6.gif">
</p>

Un colega aficionado a la economía le *sopla* que la demanda guarda una relación inversa con el precio del producto. Motivado para impresionar al querido corpóreo, se propone hacer uso de esta información para mejorar su modelo realizando las siguientes tareas:

1. Vuelva a entrenar el `Pipeline` con `XGBRegressor`, pero esta vez forzando una relación monótona negativa entre el precio y la cantidad. Para aplicar esta restricción apóyese en la siguiente <a href = https://xgboost.readthedocs.io/en/stable/tutorials/monotonic.html>documentación</a>. [6 puntos]

>Hint 1: Para implementar el constraint se le sugiere hacerlo especificando el nombre de la variable. De ser así, probablemente le sea útil **mantener el formato de pandas** antes del step de entrenamiento.

>Hint 2: Puede obtener el nombre de las columnas en el paso anterior al modelo regresor mediante el método `.get_feature_names_out()`

2. Luego, vuelva a reportar el `MAE` sobre el conjunto de validación. [1 puntos]

3. ¿Cómo cambia el error al incluir esta relación? ¿Tenía razón su amigo? [2 puntos]

4. Guarde su modelo en un archivo .pkl [1 punto]

Primero, se verán los nombres de las _features_ y su respectivo orden.

In [56]:
feature_names = pipeline.named_steps["column_transformer"].get_feature_names_out()
feature_names

array(['onehot__city_Irakleion', 'onehot__city_Larisa',
       'onehot__city_Patra', 'onehot__city_Thessaloniki',
       'onehot__day_29', 'onehot__day_30', 'onehot__day_31',
       'onehot__month_2', 'onehot__month_3', 'onehot__month_4',
       'onehot__month_5', 'onehot__month_6', 'onehot__month_7',
       'onehot__month_8', 'onehot__month_9', 'onehot__month_10',
       'onehot__month_11', 'onehot__month_12', 'onehot__year_2013',
       'onehot__year_2014', 'onehot__year_2015', 'onehot__year_2016',
       'onehot__year_2017', 'onehot__year_2018', 'onehot__shop_shop_2',
       'onehot__shop_shop_3', 'onehot__shop_shop_4',
       'onehot__shop_shop_5', 'onehot__shop_shop_6',
       'onehot__brand_gazoza', 'onehot__brand_kinder-cola',
       'onehot__brand_lemon-boost', 'onehot__brand_orange-power',
       'onehot__container_glass', 'onehot__container_plastic',
       'onehot__capacity_330ml', 'onehot__capacity_500ml',
       'remainder__id', 'remainder__lat', 'remainder__long',
       

Se busca que `remainder__price` tenga la relación negativa, entonces a dicha variable le asignaremos un $-1$ en las restricciones de monotonía.

In [57]:
if not os.path.exists("models/xgb_pipeline_monotonic.pkl"):
  # 1.
  constraints = {
    "remainder__price": -1,
  }

  constraint_list = [constraints.get(feature, 0) for feature in feature_names]
  constraint_str = "(" + ",".join(map(str, constraint_list)) + ")"

  pipeline = Pipeline(steps=[
    ("function_transformer", FunctionTransformer(func=split_date)),
    ("column_transformer", ct),
    ("regressor", XGBRegressor(monotone_constraints=constraint_str)),
  ])

  pipeline.fit(X_train, y_train)
  # 4.
  joblib.dump(pipeline, "models/xgb_pipeline_monotonic.pkl")
else:
  pipeline = joblib.load("models/xgb_pipeline_monotonic.pkl")
  print("Loaded xgb pipeline with monotonic constraints from file.")

Loaded xgb pipeline with monotonic constraints from file.


In [58]:
# 2.
y_pred_val = pipeline.predict(X_val)
mae = mean_absolute_error(y_val, y_pred_val)
print(f"Validation MAE: {mae:.2f}")

Validation MAE: 2736.99


3. ¿Cómo cambia el error al incluir esta relación? ¿Tenía razón su amigo?

> **Respuesta**: El MAE aumenta al incluir esta relación, lo que indica que sí afecta al rendimiento del modelo. Puede que hayan excepciones a esta regla que están dejándose fuera. Un ejemplo de esto es una variación muy pequeña en el precio: esto no implica necesariamente que la restricción monótona negativa se cumpla.

## 1.3 Optimización de Hiperparámetros con Optuna (20 puntos)

<p align="center">
  <img src="https://media.tenor.com/fmNdyGN4z5kAAAAi/hacking-lucy.gif">
</p>

Luego de presentarle sus resultados, Fiu le pregunta si es posible mejorar *aun más* su modelo. En particular, le comenta de la optimización de hiperparámetros con metodologías bayesianas a través del paquete `optuna`. Como usted es un aficionado al entrenamiento de modelos de ML, se propone implementar la descabellada idea de su jefe.

A partir de la mejor configuración obtenida en la sección anterior, utilice `optuna` para optimizar sus hiperparámetros. En particular, se pide que su optimización considere lo siguiente:

- Fijar una semilla en las instancias necesarias para garantizar la reproducibilidad de resultados
- Utilice `TPESampler` como método de muestreo
- De `XGBRegressor`, optimice los siguientes hiperparámetros:
    - `learning_rate` buscando valores flotantes en el rango (0.001, 0.1)
    - `n_estimators` buscando valores enteros en el rango (50, 1000)
    - `max_depth` buscando valores enteros en el rango (3, 10)
    - `max_leaves` buscando valores enteros en el rango (0, 100)
    - `min_child_weight` buscando valores enteros en el rango (1, 5)
    - `reg_alpha` buscando valores flotantes en el rango (0, 1)
    - `reg_lambda` buscando valores flotantes en el rango (0, 1)
- De `OneHotEncoder`, optimice el hiperparámetro `min_frequency` buscando el mejor valor flotante en el rango (0.0, 1.0)

Para ello se pide los siguientes pasos:
1. Implemente una función `objective()` que permita minimizar el `MAE` en el conjunto de validación. Use el método `.set_user_attr()` para almacenar el mejor pipeline entrenado. [10 puntos]
2. Fije el tiempo de entrenamiento a 5 minutos. [1 punto]
3. Optimizar el modelo y reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
> **Respuesta**: Luego de la optimización, el número de _trials_ fue $282$, el mejor MAE fue de $2.131\text{,}34$, y los mejores hiperparámetros encontrados fueron: `learning_rate`: $0\text{,}049$; `n_estimators`: $968$, `max_depth`: $7$, `max_leaves`: $81$, `min_child_weight`: $2$, `reg_alpha`: $0\text{,}228$, `reg_lambda`: $0\text{,}895$ y `min_frequency`: $0\text{,}038$.
>
> Con respecto al mejor modelo encontrado en las secciones anteriores, el MAE baja de $2.483$ a $2.131$. Esto implica una mejora, y se debe a que el modelo basal `XGBRegressor` no está optimizado para este problema en particular. Un caso típico es el desbalance de clases: a veces es necesario penalizar una clase más que otra sólo por la naturaleza de los datos. La optimización de hiperparámetros tiene esto en consideración.

4. Explique cada hiperparámetro y su rol en el modelo. ¿Hacen sentido los rangos de optimización indicados? [5 puntos]
> **Respuesta**: Los hiperparámetros seleccionados para la optimización del modelo _XGBoost_ poseen los siguientes roles:
> 1. `learning_rate`: Regula la contribución de cada árbol generado por el modelo hacia el resultado final, es decir, cuanto aprendo de cada iteración. Para este parámetro se tiene el rango ($0,1$), el rango  solicitado hace sentido dado que es bajo, permite que el modelo aprenda en buena medida, pero también permite que no exista tanto riesgo de _overfitting_ y permite experimentar sin perder estabilidad.
> 2. `n_estimators`: Número de árboles a realizar, en este caso tiene sentido que el rango sea entre ($50,1000$), ya que superior a $1000$ puede generar _overfitting_ y además sería altamente costoso computacionalmente para lo que puede ofrecer más estimadores en el caso sobre $1000$. Su rol es entregar la cantidad de árboles a generar para el modelo y aumentar la capacidad predictiva a medida que aumenta.
> 3. `max_depth`: Máxima profundidad de los árboles, regula la complejidad de los árboles a realizar, a mayor profundidad, mayor detalle que puede capturar de los datos, por tanto es bastante coherente el rango entregado, más que esa profundidad puede complejizar en exceso el modelo y generar _overfitting_.
> 4. `max_leaves`: Máxima cantidad de hojas por árbol, esto genera que se regulen las particiones que realizan los árboles, va de la mano con el hiperparámetro anterior, controlan la complejidad de los árboles del modelo. En este caso, el rango es consecuente, dado que evalúa casos donde los árboles van aumentando su complejidad.
> 5. `min_child_weight`: Peso mínimo que debe tener una hoja en el árbol, en este caso, se regula cuanta "importancia" requiere una partición para ser hoja, esto permite que el modelo sea más o menos conservador en sus particiones y decisiones. El rango entregado tiene sentido, ya que si es mayor, puede generar que el árbol no haga suficientes divisiones para predecir correctamente.
> 6. `reg_alpha`: Regulariza con L1 los pesos de los árboles, permite que los pesos estén regularizados y permite eliminar ramas del árbol que tienen poco peso. Esto permite que el árbol sea más robusto en sus pesos. En el rango solicitado, se estará probando que tan conservador debe ser para penalizar lo suficiente la complejidad del modelo y lograr balancear complejidad con funcionamiento.
> 7. `reg_lambda`: Regulariza con L2 los pesos de los árboles, posee el mismo rol, sin embargo la penalización es menor sobre la complejidad, en lugar de eliminar, regula la complejidad ya existente. El rango nuevamente hace sentido, ya que se irá probando cuanto hay que regularizar sobre los pesos para tener un árbol que pueda ser complejo, pero funcional, ambas regularizaciones evitar _overfitting_, pero una busca simplificar el árbol (L1) y otra busca generar pesos más moderados en el árbol, esto permite mejorar la robustez del modelo.
> 8. `min_frequency` : Frecuencia mínima que debe tener una categoría en una variable categórica para ser considerada una categoría propia, esto regula que el _encoder_ separe las categorias de forma significativa y previene que la dimensionalidad suba en exceso por categorías que aparecen pocas veces (como lo pueden ser _outliers_). El rango es coherente, a mayor frecuencia mínima, más se le exige para ser considerado una categoría, entonces va regulando entre permitir todas o exigir más frecuencia.
>
> Fuentes: [Documentación XGBoost](https://xgboost.readthedocs.io/en/stable/parameter.html), [Documentación XGBoost Python API](https://xgboost.readthedocs.io/en/stable/python/python_api.html#xgboost.XGBRegressor)

5. Guardar su modelo en un archivo .pkl [1 punto]

In [59]:
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1.
def objective(trial) -> float:
    """
    Objective function for Optuna hyperparameter optimization.
    
    Parameters
    ----------
    trial : optuna.trial.Trial
        A trial object that is used to suggest hyperparameters.

    Returns
    -------
    float
        The mean absolute error (MAE) on the validation set.
    """
    learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1, log=True)
    n_estimators = trial.suggest_int("n_estimators", 50, 1000)
    max_depth = trial.suggest_int("max_depth", 3, 10)
    max_leaves = trial.suggest_int("max_leaves", 0, 100)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 5)
    reg_alpha = trial.suggest_float("reg_alpha", 0.0, 1.0)
    reg_lambda = trial.suggest_float("reg_lambda", 0.0, 1.0)
    min_frequency = trial.suggest_float("min_frequency", 0.0, 1.0)

    ct = ColumnTransformer(transformers=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first",
                sparse_output=False,
                min_frequency=min_frequency
            ),
            categorical
        ),
    ], remainder="passthrough")

    model = XGBRegressor(
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_leaves=max_leaves,
        min_child_weight=min_child_weight,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    pipeline = Pipeline(steps=[
        ("function_transformer", FunctionTransformer(func=split_date)),
        ("column_transformer", ct),
        ("regressor", model),
    ])

    pipeline.fit(X_train, y_train)
    y_pred_val = pipeline.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred_val)
    
    trial.set_user_attr("pipeline", pipeline)
    return mae

In [60]:
if not os.path.exists("models/xgb_pipeline_optuna.pkl"):
  # 2.
  sampler = TPESampler(seed=RANDOM_STATE)
  study = optuna.create_study(direction="minimize", sampler=sampler)

  study.optimize(objective, timeout=300)

  print("Número de trials:", len(study.trials))
  print("Mejor MAE:", study.best_value)
  print("Mejores hiperparámetros:")
  for k, v in study.best_params.items():
    print(f"  {k}: {v}")

  best_pipeline = study.best_trial.user_attrs["pipeline"]
  # 5.
  joblib.dump(best_pipeline, "models/xgb_pipeline_optuna.pkl")
else:
  best_pipeline = joblib.load("models/xgb_pipeline_optuna.pkl")
  print("Loaded xgb pipeline with optuna from file.")

Loaded xgb pipeline with optuna from file.


## 4. Optimización de Hiperparámetros con Optuna y Prunners (17 puntos)

<p align="center">
  <img src="https://i.pinimg.com/originals/90/16/f9/9016f919c2259f3d0e8fe465049638a7.gif">
</p>

Después de optimizar el rendimiento de su modelo varias veces, Fiu le pregunta si no es posible optimizar el entrenamiento del modelo en sí mismo. Después de leer un par de post de personas de dudosa reputación en la *deepweb*, usted llega a la conclusión que puede cumplir este objetivo mediante la implementación de **Prunning**.

Vuelva a optimizar los mismos hiperparámetros que la sección pasada, pero esta vez utilizando **Prunning** en la optimización. En particular, usted debe:

- Responder: ¿Qué es prunning? ¿De qué forma debería impactar en el entrenamiento? [2 puntos]
> **Respuesta**: La técnica de _pruning_, en el contexto de Optuna, hace referencia a la aplicación de _early stopping_ durante la ejecución de un _trial_. Aplicar _early stopping_ significa parar el entrenamiento cuando se ve que el rendimiento entre épocas (medido por entrenamiento y validación en la métrica de interés) ya no está mejorando. Un _trial_ está compuesto de varias épocas.
>
> El impacto que tiene es principalmente en la eficiencia: permite **explorar** más combinaciones de hiperparámetros en el mismo tiempo de ejecución, dado que los recursos se liberan antes. Esto puede mejorar la calidad del resultado final, sin embargo, si el _pruner_ es muy agresivo (mala implementación de _early stopping_), pueden detenerse _trials_ que sí hubiese mejorado, afectando la calidad global.
>
> Es por esto que es necesario configurar bien los hiperparámetros `n_startup_trials`: número de trials sin _pruning_ al inicio del estudio, `n_warmup_steps`: número mínimo de épocas que deben transcurrir antes de aplicar _pruning_, y el tipo de _pruner_: `MedianPruner`, `SuccessiveHalvingPruner`, etc.
- Redefinir la función `objective()` utilizando `optuna.integration.XGBoostPruningCallback` como método de **Prunning** [10 puntos]
- Fijar nuevamente el tiempo de entrenamiento a 5 minutos [1 punto]
- Reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
> **Respuesta**: El número de _trials_ fue de $242$, el mejor MAE fue de $2.109$, y los mejores hiperparámetros fueron: `learning_rate`: $0\text{,}049$, `n_estimators`: $864$, `max_depth`: $7$, `max_leaves`: $75$, `min_child_weight`: $2$, `reg_alpha`: $0\text{,}403$, `reg_lambda`: $0\text{,}679$ y `min_frequency`: $0\text{,}019$. Con respecto al mejor resultado de la parte anterior (_Optuna sin pruning_), la métrica MAE baja de $2.131\text{,}34$ a $2.109$. 
>
> Esto se puede deber a lo justificado en el primer punto: el implementar _early stopping_ permite explorar las combinaciones de hiperparámetros que realmente generan modelos con buen rendimiento.
- Guardar su modelo en un archivo .pkl [1 punto]

Nota: Si quieren silenciar los prints obtenidos en el prunning, pueden hacerlo mediante el siguiente comando:

```
optuna.logging.set_verbosity(optuna.logging.WARNING)
```

De implementar la opción anterior, pueden especificar `show_progress_bar = True` en el método `optimize` para *más sabor*.

Hint: Si quieren especificar parámetros del método .fit() del modelo a través del pipeline, pueden hacerlo por medio de la siguiente sintaxis: `pipeline.fit(stepmodelo__parametro = valor)`

Hint2: Este <a href = https://stackoverflow.com/questions/40329576/sklearn-pass-fit-parameters-to-xgboost-in-pipeline>enlace</a> les puede ser de ayuda en su implementación

In [86]:
from xgboost.callback import EarlyStopping

# 1.
def objective(trial) -> float:
    """
    Objective function for Optuna hyperparameter optimization.
    This function uses pruning to stop unpromising trials early.
    
    Parameters
    ----------
    trial : optuna.trial.Trial
        A trial object that is used to suggest hyperparameters.

    Returns
    -------
    float
        The mean absolute error (MAE) on the validation set.
    """
    learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1, log=True)
    n_estimators = trial.suggest_int("n_estimators", 50, 1000)
    max_depth = trial.suggest_int("max_depth", 3, 10)
    max_leaves = trial.suggest_int("max_leaves", 0, 100)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 5)
    reg_alpha = trial.suggest_float("reg_alpha", 0.0, 1.0)
    reg_lambda = trial.suggest_float("reg_lambda", 0.0, 1.0)
    min_frequency = trial.suggest_float("min_frequency", 0.0, 1.0)

    preprocessor = Pipeline([
        ("function_transformer", FunctionTransformer(func=split_date)),
        ("column_transformer", ColumnTransformer([
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    drop="first",
                    sparse_output=False,
                    min_frequency=min_frequency
                ),
                categorical
            ),
        ], remainder="passthrough")),
    ])

    X_train_prepared = preprocessor.fit_transform(X_train)
    X_val_prepared = preprocessor.transform(X_val)

    pruning_callback = optuna.integration.XGBoostPruningCallback(
        trial, 
        observation_key="validation_0-mae"
    )

    model = XGBRegressor(
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_leaves=max_leaves,
        min_child_weight=min_child_weight,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
        eval_metric="mae",
        early_stopping_rounds=50,
        callbacks=[pruning_callback]
    )

    model.fit(
        X_train_prepared, 
        y_train,
        eval_set=[(X_val_prepared, y_val)],
        verbose=False
    )

    y_pred_val = model.predict(X_val_prepared)
    mae = mean_absolute_error(y_val, y_pred_val)

    trial.set_user_attr("model", model)
    return mae

In [82]:
if not os.path.exists("models/xgb_model_optuna_pruning.pkl"):
  # 2.
  sampler = TPESampler(seed=RANDOM_STATE)
  study = optuna.create_study(direction="minimize", sampler=sampler)

  study.optimize(objective, timeout=300)

  print("Número de trials:", len(study.trials))
  print("Mejor MAE:", study.best_value)
  print("Mejores hiperparámetros:")
  for k, v in study.best_params.items():
    print(f"  {k}: {v}")

  model = study.best_trial.user_attrs["model"]
  # 4.
  joblib.dump(model, "models/xgb_model_optuna_pruning.pkl")
else:
  model = joblib.load("models/xgb_model_optuna_pruning.pkl")
  print("Loaded xgb pipeline with optuna + pruning from file.")

Número de trials: 242
Mejor MAE: 2109.001609331801
Mejores hiperparámetros:
  learning_rate: 0.04983812211147949
  n_estimators: 864
  max_depth: 7
  max_leaves: 75
  min_child_weight: 2
  reg_alpha: 0.40327581782398975
  reg_lambda: 0.6792748782148914
  min_frequency: 0.0192274581115719


## 5. Visualizaciones (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/F-LgB1xTebEAAAAd/look-at-this-graph-nickelback.gif">
</p>


Satisfecho con su trabajo, Fiu le pregunta si es posible generar visualizaciones que permitan entender el entrenamiento de su modelo.

A partir del siguiente <a href = https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/005_visualization.html#visualization>enlace</a>, genere las siguientes visualizaciones:

1. Gráfico de historial de optimización [1 punto]
2. Gráfico de coordenadas paralelas [1 punto]
3. Gráfico de importancia de hiperparámetros [1 punto]

Comente sus resultados:

4. ¿Desde qué *trial* se empiezan a observar mejoras notables en sus resultados? [0.5 puntos]
> **Respuesta**: A partir del _trial_ $0$ se ve una baja significativa, que se repite nuevamente en el _trial_ $12$. Una baja significativa en este contexto significa una mejora notable. Desde el _trial_ $15$ se estabilizan los resultados.
5. ¿Qué tendencias puede observar a partir del gráfico de coordenadas paralelas? [1 punto]
> **Respuesta**: Para el análisis, se considerarán sólo los $3$ hiperparámetros de mayor importancia del siguiente ítem. Las líneas más oscuras (menores MAE) están concentradas en la parte baja del eje de `learning_rate`. Esto sugiere que valores bajos de este hiperparámetro $0\text{,}001$-$0\text{,}01$ dieron los mejores resultados. Estas líneas tienden a ubicarse en la zona media del eje ($\sim 0\text{,}3$–$0\text{,}6$) para `min_frequency`, lo que indica que este rango es el más óptimo para estos hiperparámetros. Por último, con `max_leaves`, el modelo parece funcionar mejor con una complejidad intermedia (entre $20$ y $70$ hojas).
6. ¿Cuáles son los hiperparámetros con mayor importancia para la optimización de su modelo? [0.5 puntos]
> **Respuesta**: El hiperparámetro más importante para la optimización del modelo es `min_frequency`, por lejos, que representa la frecuencia mínima que debe tener una categoría en una variable categórica para ser considerada una categoría propia. Lo siguen `learning_rate`, la tasa de aprendizaje del modelo, y `max_leaves`, el número máximo de hojas que puede tener un árbol. 

Para realizar los gráficos, se usará el último estudio (Optuna + _pruning_), dado que entregó el mejor resultado para MAE.

In [84]:
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_param_importances

fig = plot_optimization_history(study)
fig.show()

fig = plot_parallel_coordinate(study)
fig.show()

fig = plot_param_importances(study)
fig.show()

## 6. Síntesis de resultados (3 puntos)

Finalmente:

1. Genere una tabla resumen del MAE en el conjunto de validación obtenido en los 5 modelos entrenados desde Baseline hasta XGBoost con Constraints, Optuna y Prunning. [1 punto]
> **Respuesta**: 
> $$
> \begin{array}{|c|c|}
> \hline
> \textbf{Modelo} & \textbf{MAE en validación} \\
> \hline
> \textit{Baseline} & 13.543\text{,}96 \\
> \text{XGBoost basal} & 2.483\text{,}77 \\
> \text{XGBoost con \textit{constraints}} & 2.736\text{,}99 \\
> \text{XGBoost + Optuna} & 2.131\text{,}34 \\
> \text{XGBoost + Optuna + \textit{Pruning}} & 2.109 \\
> \hline
> \end{array}
> $$
2. Compare los resultados de la tabla y responda, ¿qué modelo obtiene el mejor rendimiento? [0.5 puntos]
> **Respuesta**: El modelo que obtiene el mejor rendimiento es XGBoost + Optuna + _Pruning_. Esto se debe a los motivos ya justificados: además de optimizar los hiperparámetros de XGBoost, se preocupa de explorar las combinaciones que más aportan a minimizar la métrica MAE.
3. Cargue el mejor modelo, prediga sobre el conjunto de **test** y reporte su MAE. [0.5 puntos]
4. ¿Existen diferencias con respecto a las métricas obtenidas en el conjunto de validación? ¿Porqué puede ocurrir esto? [1 punto]
> **Respuesta**: Sí, existen diferencias con respecto al conjunto de validación. En particular, el MAE en _test_ es menor que el MAE en validación. Esto se puede deber a que el conjunto de _test_ tiene datos que fueron mejor capturados en el entrenamiento, a pesar de que la optimización de hiperparámetros se hiciera con el conjunto de validación. Puede indicar también que existen similitudes entre estos dos conjuntos de datos.

In [ ]:
# 3.
optimized_model = joblib.load("models/xgb_model_optuna_pruning.pkl")

preprocessor = Pipeline([
  ("function_transformer", FunctionTransformer(func=split_date)),
  ("column_transformer", ColumnTransformer([
    (
      "onehot",
      OneHotEncoder(
        handle_unknown="ignore",
        drop="first",
        sparse_output=False,
        min_frequency=study.best_params["min_frequency"]
      ),
      categorical
    ),
  ], remainder="passthrough")),
])

preprocessor.fit(X_train)
X_test_prepared = preprocessor.transform(X_test)
y_pred_test = optimized_model.predict(X_test_prepared)
mae_test = mean_absolute_error(y_test, y_pred_test)
print(f"Test MAE: {mae_test:.2f}")

Test MAE: 2038.76


# Conclusión
Exito!
<p align="center">
  <img src="https://i.pinimg.com/originals/55/3d/42/553d42bea9b10e0662a05aa8726fc7f4.gif">
</p>